# Scalar Subqueries

## Overview
A **scalar subquery** is a subquery that returns exactly **one row and one column** — a single value. It can appear anywhere a single value is allowed: in SELECT, WHERE, HAVING, or even ORDER BY.

```sql
-- In SELECT: compute a value relative to the whole table
SELECT enc_id, cost_cad,
       (SELECT AVG(cost_cad) FROM encounters) AS overall_avg
FROM encounters;

-- In WHERE: filter against a computed threshold
SELECT * FROM encounters
WHERE cost_cad > (SELECT AVG(cost_cad) FROM encounters);

-- In HAVING: filter a group against a global aggregate
SELECT dept_id, AVG(cost_cad)
FROM encounters
GROUP BY dept_id
HAVING AVG(cost_cad) > (SELECT AVG(cost_cad) FROM encounters);
```

**Rules:**
- Must return exactly one row and one column — if it returns more, the query errors
- If it returns zero rows, the result is NULL
- Executed once (non-correlated) or once per outer row (correlated — see next notebook)
- Can reference outer query tables if correlated (covered in `correlated_subqueries`)

**When to use vs alternatives:**
- Scalar subquery in SELECT → often replaceable by a window function (more efficient)
- Scalar subquery in WHERE → often replaceable by a JOIN or CTE
- For a single global reference value (overall avg, max date), scalar subqueries are clean and readable

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (
    patient_id  INTEGER PRIMARY KEY,
    first_name  TEXT, last_name TEXT,
    dob         TEXT, sex TEXT, province TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE providers (
    provider_id INTEGER PRIMARY KEY,
    full_name   TEXT, specialty TEXT, province TEXT, manager_id INTEGER
);
CREATE TABLE departments (
    dept_id     INTEGER PRIMARY KEY,
    dept_name   TEXT, building TEXT, head_id INTEGER
);
CREATE TABLE encounters (
    enc_id      INTEGER PRIMARY KEY,
    patient_id  INTEGER, provider_id INTEGER, dept_id INTEGER,
    enc_date    TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER
);
CREATE TABLE diagnoses (
    diag_code TEXT PRIMARY KEY, description TEXT, category TEXT
);
CREATE TABLE lab_results (
    result_id  INTEGER PRIMARY KEY,
    patient_id INTEGER, test_name TEXT,
    result_val REAL, units TEXT, collected TEXT, flagged INTEGER
);

INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','F','NB',1),
  (2,'Liam','Chen','1972-11-04','M','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),
  (4,'James','MacLeod','1955-01-30','M','NB',0),
  (5,'Sofia','Petrov','2001-09-15','F','BC',1),
  (6,'Noah','Williams','1968-06-08','M','AB',1),
  (7,'Mei','Zhang','1995-04-17','F','ON',1),
  (8,'Ethan','Tremblay','1980-12-01','M','QC',0),
  (9,'Priya','Nair','1977-08-25','F','BC',1),
  (10,'Marcus','Okafor','1993-05-19','M','ON',1),
  (11,'Diana','Flores','2000-02-14','F','NB',1);

INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),
  (11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Osei','Family Medicine','ON',10),
  (13,'Dr. Ethan Larson','Orthopaedics','QC',10),
  (14,'Dr. Priya Sharma','Emergency','AB',10),
  (15,'Dr. Lucas Petit','Radiology','QC',11);

INSERT INTO departments VALUES
  (1,'Emergency','Tower A',14),(2,'Cardiology','Tower B',10),
  (3,'Oncology','Tower C',11),(4,'Family Medicine','Clinic',12),
  (5,'Orthopaedics','Tower A',13),(6,'Radiology','Tower B',15),
  (7,'ICU','Tower C',NULL);

INSERT INTO encounters VALUES
  (1, 1,10,2,'2023-01-15','I25.1',3200.00,1),
  (2, 2,14,1,'2023-02-20','J18.9',1850.00,1),
  (3, 3,12,4,'2023-03-05','Z00.0', 120.00,0),
  (4, 4,13,5,'2023-03-18','M17.1',5500.00,1),
  (5, 5,12,4,'2023-04-02','J06.9',  95.00,0),
  (6, 6,14,1,'2023-05-11','R07.9', 780.00,0),
  (7, 7,10,2,'2023-06-22','I10',  2100.00,1),
  (8, 8,12,4,'2023-07-14',NULL,    80.00,0),
  (9, 1,14,1,'2023-08-30','R55',  410.00,0),
  (10,3,12,4,'2023-09-12','Z00.0', 110.00,0),
  (11,9,10,2,'2023-10-01','I48.0',1750.00,1),
  (12,10,14,1,'2023-11-03','S52.5',2200.00,1),
  (13,2,10,2,'2023-11-20','I25.1',2900.00,1),
  (14,6,12,4,'2023-12-01','Z00.0', 115.00,0);

INSERT INTO diagnoses VALUES
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),
  ('J18.9','Pneumonia, unspecified','Respiratory'),
  ('Z00.0','General medical examination','Preventive'),
  ('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),
  ('R07.9','Chest pain, unspecified','Symptoms'),
  ('I10',  'Essential hypertension','Cardiovascular'),
  ('R55',  'Syncope and collapse','Symptoms'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),
  ('S52.5','Fracture of lower end of radius','Injury'),
  ('E11.9','Type 2 diabetes mellitus','Endocrine'),
  ('I50.9','Heart failure, unspecified','Cardiovascular');

INSERT INTO lab_results VALUES
  (1, 1,'HbA1c',     7.2,'%',      '2023-03-10',0),
  (2, 2,'HbA1c',     9.1,'%',      '2023-03-15',1),
  (3, 3,'Creatinine',88.5,'umol/L','2023-04-01',0),
  (4, 4,'Creatinine',145.0,'umol/L','2023-04-12',1),
  (5, 5,'HbA1c',     5.8,'%',      '2023-05-05',0),
  (6, 6,'Sodium',   138.0,'mmol/L','2023-05-20',0),
  (7, 7,'Sodium',   151.0,'mmol/L','2023-06-01',1),
  (8, 1,'Creatinine',72.3,'umol/L','2023-06-18',0),
  (9, 8,'HbA1c',     6.4,'%',      '2023-07-02',0),
  (10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),
  (11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),
  (12,10,'HbA1c',    7.8,'%',      '2023-08-20',1);
""")
conn.commit()

print('Healthcare schema ready. Table row counts:')
for t in ['patients','providers','departments','encounters','diagnoses','lab_results']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n} rows')


---
## Scalar subquery in SELECT — appending a global reference value

In [ ]:
# Each encounter with the overall average cost appended
print('=== Each encounter vs overall average cost ===')
q = """
SELECT  enc_id,
        patient_id,
        enc_date,
        cost_cad,
        ROUND((SELECT AVG(cost_cad) FROM encounters), 2)        AS overall_avg,
        ROUND(cost_cad - (SELECT AVG(cost_cad) FROM encounters), 2) AS diff_from_avg,
        CASE WHEN cost_cad > (SELECT AVG(cost_cad) FROM encounters)
             THEN 'Above avg' ELSE 'Below avg' END              AS vs_avg
FROM    encounters
ORDER BY cost_cad DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Scalar subquery in WHERE — filtering against a derived threshold

In [ ]:
# Encounters costing more than the average
print('=== Encounters above average cost ===')
q = """
SELECT  enc_id, patient_id, enc_date, diag_code, cost_cad
FROM    encounters
WHERE   cost_cad > (SELECT AVG(cost_cad) FROM encounters)
ORDER BY cost_cad DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Most recent encounter date per patient — use scalar subquery to compare
print()
print('=== Patients whose last encounter was before 2023-06-01 ===')
q2 = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name  AS patient,
        (SELECT MAX(enc_date)
         FROM   encounters
         WHERE  patient_id = p.patient_id)  AS last_enc_date
FROM    patients AS p
WHERE   (SELECT MAX(enc_date)
         FROM   encounters
         WHERE  patient_id = p.patient_id) < '2023-06-01'
  AND   (SELECT MAX(enc_date)
         FROM   encounters
         WHERE  patient_id = p.patient_id) IS NOT NULL
ORDER BY last_enc_date
"""
print(pd.read_sql(q2, conn).to_string(index=False))
print()
print('Note: the repeated subquery is inefficient — a CTE or window function is better.')
print('This is shown here to demonstrate the pattern; see basic_ctes for the cleaner version.')


---
## Scalar subquery in HAVING

In [ ]:
# Departments whose average encounter cost exceeds the hospital-wide average
print('=== Departments with above-average encounter cost ===')
q = """
SELECT  d.dept_name,
        COUNT(e.enc_id)            AS encounters,
        ROUND(AVG(e.cost_cad), 2)  AS dept_avg_cost,
        ROUND((SELECT AVG(cost_cad) FROM encounters), 2) AS hospital_avg
FROM    encounters   AS e
JOIN    departments  AS d ON e.dept_id = d.dept_id
GROUP BY d.dept_id, d.dept_name
HAVING  AVG(e.cost_cad) > (SELECT AVG(cost_cad) FROM encounters)
ORDER BY dept_avg_cost DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Single-value lookups and the zero-row NULL case

In [ ]:
# Scalar subquery returning NULL when no rows match
print('=== Scalar subquery returns NULL when subquery has no matching rows ===')
q = """
SELECT  p.patient_id,
        p.first_name || ' ' || p.last_name    AS patient,
        -- patients with no encounters: subquery returns NULL
        (SELECT MAX(enc_date)
         FROM   encounters
         WHERE  patient_id = p.patient_id)    AS last_enc_date,
        COALESCE(
            (SELECT MAX(enc_date)
             FROM   encounters
             WHERE  patient_id = p.patient_id),
            '(no encounters)'
        )                                      AS last_enc_safe
FROM    patients AS p
ORDER BY last_enc_date NULLS LAST
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Scalar subquery to bring in a single reference value from another table
print()
print('=== Latest lab collection date across all patients ===')
q2 = """
SELECT  result_id, patient_id, test_name, result_val,
        collected,
        (SELECT MAX(collected) FROM lab_results)  AS latest_collection,
        CAST(JULIANDAY((SELECT MAX(collected) FROM lab_results))
             - JULIANDAY(collected) AS INTEGER)   AS days_before_latest
FROM    lab_results
ORDER BY collected
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Subquery returns more than one row — query errors**
`WHERE cost_cad = (SELECT cost_cad FROM encounters WHERE admitted = 1)` fails if more than one admitted encounter exists. Use `MAX`, `MIN`, `AVG`, or another aggregate to guarantee one row, or use `IN` / `EXISTS` for multi-row subqueries.

**2. Subquery returns zero rows — result is NULL, not zero**
`(SELECT AVG(cost_cad) FROM encounters WHERE dept_id = 999)` returns NULL because dept 999 doesn't exist. `cost_cad - NULL` is NULL. Wrap with `COALESCE` when a fallback value is needed.

**3. Repeating the same scalar subquery in SELECT and WHERE**
Running the same subquery twice — once in SELECT and once in WHERE — means it executes twice. Use a CTE to compute it once and reference the result in both places. Most query optimisers will deduplicate simple cases, but explicit CTEs are clearer and more portable.

**4. Scalar subquery in SELECT can be slower than a window function**
`(SELECT AVG(cost_cad) FROM encounters)` in SELECT is computed once (non-correlated) and is fine. But `(SELECT AVG(cost_cad) FROM encounters WHERE dept_id = e.dept_id)` is correlated — it executes once per outer row. The equivalent `AVG(cost_cad) OVER (PARTITION BY dept_id)` is far more efficient for per-group reference values.

**5. Forgetting that scalar subqueries execute in the context of their containing query**
A scalar subquery in SELECT is evaluated row-by-row conceptually. If it references a column from the outer query (correlated), it re-executes for every row. Non-correlated subqueries (no outer reference) execute once. Understanding this distinction is key to predicting performance.

**6. Using a scalar subquery where a join would be clearer**
`SELECT (SELECT dept_name FROM departments WHERE dept_id = e.dept_id)` works but obscures the relationship. A simple JOIN is more readable, more maintainable, and allows the optimiser more flexibility. Reserve scalar subqueries for cases where a join would change cardinality or where the value genuinely isn't a join relationship.


---
*sql_methods_library - Samantha McGarrigle*